In [1]:
import torch
from flow_matching.supervised.alphas_betas import LinearAlpha, LinearBeta
from flow_matching.supervised.prob_paths import GaussianCondProbPath
from flow_matching.training.flow_trainer import FlowTrainer
from flow_matching.mnist.mnist_sampler import MNISTSampler
from flow_matching.architectures.resunet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
num_classes = 10

path = GaussianCondProbPath(
    p_data=MNISTSampler(),
    p_simple_shape=(1, 32, 32),
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
)

trainer = FlowTrainer(
    path=path,
    model=backbone,
    eta=1 / (num_classes + 1),
    null_class=num_classes,
    num_classes=num_classes,
)

In [4]:
trainer.train(
    num_epochs=50,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=50,
    validate=False,
)

2025-10-05 18:28:46,242 - flow-matching - INFO - Training model with size: 2.734 MiB
Epoch 49/50: 100%|██████████| 50/50 [00:09<00:00,  5.45it/s, loss=0.135576]


In [5]:
torch.save(backbone.state_dict(), "backbone.pt")